# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshithavalli1006-spec/FlyRank--AI-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content item for one client. The analysis uses a 90-day historical window, with the most recent 30 days used for outcome signals and the previous 30 days used for comparison features.

In [4]:
import os
import getpass
import duckdb
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected
Feature window: February 2026
Label window: March 2026


In [5]:

print("February 2026 data schema:")
print(con.execute(f"DESCRIBE SELECT * FROM {FEB} LIMIT 1").fetchdf())

February 2026 data schema:
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  No

In [6]:
print("Unit of analysis verification")

print("\nTotal rows:")
print(con.execute(f"SELECT COUNT(*) FROM {FEB}").fetchone()[0])

print("\nDate range:")
print(con.execute(f"""
SELECT MIN(report_date), MAX(report_date)
FROM {FEB}
""").fetchone())

print("\nUnique client-content pairs:")
print(con.execute(f"""
SELECT COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
FROM {FEB}
""").fetchone()[0])

print("\nDuplicate client-content-date rows:")
print(con.execute(f"""
SELECT COUNT(*)
FROM (
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
)
""").fetchone()[0])

Unit of analysis verification

Total rows:
7355108

Date range:
(datetime.date(2026, 2, 1), datetime.date(2026, 2, 28))

Unique client-content pairs:
321546

Duplicate client-content-date rows:
0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: imp_prev30, visible_queries, rare_share, anon_share, top_query_share, position_volatility.
Label: is_declining, meaning impressions declined by more than 20% in the last 30 days compared with the previous 30 days.
Context: client_hash_id, content_hash_id, report_date, and query-level signals used to understand the content and client.
Excluded: raw identifiers and fields that are not available before the prediction window or could cause target leakage.

In [7]:
required_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

columns = con.execute(f"""
SELECT column_name
FROM (
    DESCRIBE SELECT * FROM {FEB}
)
""").fetchdf()["column_name"].tolist()

print("Required fields:")
for field in required_fields:
    print(field, "✓" if field in columns else "✗")

print("\nExcluded raw identifiers: client_hash_id, content_hash_id")


Required fields:
report_date ✓
client_hash_id ✓
content_hash_id ✓
gsc_impressions ✓
gsc_clicks ✓
gsc_avg_position ✓

Excluded raw identifiers: client_hash_id, content_hash_id


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
print("Missing values in February 2026:")

print(con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_date,
    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_gsc_position
FROM {FEB}
""").fetchdf())

Missing values in February 2026:
   total_rows  missing_client  missing_content  missing_date  \
0     7355108               0                0             0   

   missing_gsc_impressions  missing_gsc_position  
0                    92256               4733326  


In [9]:
print("\nMissing-value percentages:")

result = con.execute(f"""
SELECT
    ROUND(100.0 * COUNT(*) FILTER (WHERE client_hash_id IS NULL) / COUNT(*), 2) AS client_missing_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE content_hash_id IS NULL) / COUNT(*), 2) AS content_missing_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE report_date IS NULL) / COUNT(*), 2) AS date_missing_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_impressions IS NULL) / COUNT(*), 2) AS impressions_missing_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) / COUNT(*), 2) AS position_missing_pct
FROM {FEB}
""").fetchdf()

print(result)


Missing-value percentages:
   client_missing_pct  content_missing_pct  date_missing_pct  \
0                 0.0                  0.0               0.0   

   impressions_missing_pct  position_missing_pct  
0                     1.25                 64.35  


In [10]:
print("\nFebruary 2026 window:")
print(con.execute(f"""
SELECT MIN(report_date) AS start_date,
       MAX(report_date) AS end_date,
       COUNT(DISTINCT report_date) AS days
FROM {FEB}
""").fetchdf())

print("\nMarch 2026 label window:")
print(con.execute(f"""
SELECT MIN(report_date) AS start_date,
       MAX(report_date) AS end_date,
       COUNT(DISTINCT report_date) AS days
FROM {MAR}
""").fetchdf())


February 2026 window:
  start_date   end_date  days
0 2026-02-01 2026-02-28    28

March 2026 label window:
  start_date   end_date  days
0 2026-03-01 2026-03-31    31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits: The data may have unbalanced history across content items, especially for early rows. Some early observations may rely only on GSC data, and overlapping feature/outcome windows can make comparisons less independent. These limitations should be considered when interpreting model results.

In [12]:
print("Data limits check")

print("\nRows in February:")
print(con.execute(f"SELECT COUNT(*) FROM {FEB}").fetchone()[0])

print("\nRows in March:")
print(con.execute(f"SELECT COUNT(*) FROM {MAR}").fetchone()[0])

print("\nClients in February:")
print(con.execute(f"""
SELECT COUNT(DISTINCT client_hash_id)
FROM {FEB}
""").fetchone()[0])

print("\nContent items in February:")
print(con.execute(f"""
SELECT COUNT(DISTINCT content_hash_id)
FROM {FEB}
""").fetchone()[0])

Data limits check

Rows in February:
7355108

Rows in March:
9841378

Clients in February:
54

Content items in February:
321546


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.